In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder.appName("ChipSeqExample").enableHiveSupport().getOrCreate()

In [4]:
# Example ChIP-seq metadata
data = [
    ("S1", "CTCF",   "K562",     "rep1", 120345),
    ("S2", "CTCF",   "K562",     "rep2", 118932),
    ("S3", "CTCF",   "GM12878",  "rep1", 113221),
    ("S4", "REST",   "K562",     "rep1",  54321),
    ("S5", "REST",   "GM12878",  "rep1",  49876),
    ("S6", "GATA1",  "K562",     "rep1",  87654),
    ("S7", "GATA1",  "K562",     "rep2",  90231),
    ("S8", "GATA1",  "GM12878",  "rep1",  81234),
]

columns = ["sample_id", "TF", "cell_type", "replicate", "num_peaks"]

In [7]:
df = spark.createDataFrame(data, columns)
df.show(5)

+---------+----+---------+---------+---------+
|sample_id|  TF|cell_type|replicate|num_peaks|
+---------+----+---------+---------+---------+
|       S1|CTCF|     K562|     rep1|   120345|
|       S2|CTCF|     K562|     rep2|   118932|
|       S3|CTCF|  GM12878|     rep1|   113221|
|       S4|REST|     K562|     rep1|    54321|
|       S5|REST|  GM12878|     rep1|    49876|
+---------+----+---------+---------+---------+
only showing top 5 rows


In [8]:
# Save as Hive table
df.write.mode("append").saveAsTable("chipseq_metadata")

In [9]:
# Query Hive table
hive_df = spark.table("chipseq_metadata")

In [10]:
# Aggregation 1 — Total number of peaks per TF
peaks_per_tf = (
    hive_df.groupBy("TF")
      .agg(F.sum("num_peaks").alias("total_peaks"))
      .orderBy(F.desc("total_peaks"))
)

peaks_per_tf.show()

+-----+-----------+
|   TF|total_peaks|
+-----+-----------+
| CTCF|     352498|
|GATA1|     259119|
| REST|     104197|
+-----+-----------+



In [11]:
# Aggregation 2 — Average peaks per TF and cell type
avg_peaks_tf_cell = (
    hive_df.groupBy("TF", "cell_type")
      .agg(F.avg("num_peaks").alias("avg_peaks"))
      .orderBy("TF", "cell_type")
)

avg_peaks_tf_cell.show()

+-----+---------+---------+
|   TF|cell_type|avg_peaks|
+-----+---------+---------+
| CTCF|  GM12878| 113221.0|
| CTCF|     K562| 119638.5|
|GATA1|  GM12878|  81234.0|
|GATA1|     K562|  88942.5|
| REST|  GM12878|  49876.0|
| REST|     K562|  54321.0|
+-----+---------+---------+



In [12]:
# Aggregation 3 — Replicate consistency (min / max / std)
replicate_stats = (
    hive_df.groupBy("TF", "cell_type")
      .agg(
          F.count("*").alias("n_replicates"),
          F.min("num_peaks").alias("min_peaks"),
          F.max("num_peaks").alias("max_peaks"),
          F.stddev("num_peaks").alias("std_peaks")
      )
)

replicate_stats.show()

+-----+---------+------------+---------+---------+-----------------+
|   TF|cell_type|n_replicates|min_peaks|max_peaks|        std_peaks|
+-----+---------+------------+---------+---------+-----------------+
|GATA1|     K562|           2|    87654|    90231|1822.214175117733|
| REST|  GM12878|           1|    49876|    49876|             NULL|
|GATA1|  GM12878|           1|    81234|    81234|             NULL|
| CTCF|  GM12878|           1|   113221|   113221|             NULL|
| REST|     K562|           1|    54321|    54321|             NULL|
| CTCF|     K562|           2|   118932|   120345|999.1418818165916|
+-----+---------+------------+---------+---------+-----------------+



In [13]:
# Aggregation 4 — Flag low-quality samples
qc_flagged = hive_df.withColumn(
    "low_quality",
    F.col("num_peaks") < 60000
)

qc_flagged.show()

+---------+-----+---------+---------+---------+-----------+
|sample_id|   TF|cell_type|replicate|num_peaks|low_quality|
+---------+-----+---------+---------+---------+-----------+
|       S5| REST|  GM12878|     rep1|    49876|       true|
|       S6|GATA1|     K562|     rep1|    87654|      false|
|       S7|GATA1|     K562|     rep2|    90231|      false|
|       S8|GATA1|  GM12878|     rep1|    81234|      false|
|       S1| CTCF|     K562|     rep1|   120345|      false|
|       S2| CTCF|     K562|     rep2|   118932|      false|
|       S3| CTCF|  GM12878|     rep1|   113221|      false|
|       S4| REST|     K562|     rep1|    54321|       true|
+---------+-----+---------+---------+---------+-----------+



In [14]:
low_qc_summary = (
    qc_flagged.groupBy("TF")
      .agg(F.sum(F.col("low_quality").cast("int")).alias("low_qc_samples"))
)

low_qc_summary.show()

+-----+--------------+
|   TF|low_qc_samples|
+-----+--------------+
| REST|             2|
|GATA1|             0|
| CTCF|             0|
+-----+--------------+



In [15]:
# Aggregation 5 — Percentage of peaks per TF
total_peaks = hive_df.agg(F.sum("num_peaks")).collect()[0][0]

fraction_per_tf = (
    hive_df.groupBy("TF")
      .agg(
          (F.sum("num_peaks") / F.lit(total_peaks)).alias("fraction_of_total")
      )
      .orderBy(F.desc("fraction_of_total"))
)

fraction_per_tf.show(truncate=False)

+-----+-------------------+
|TF   |fraction_of_total  |
+-----+-------------------+
|CTCF |0.49244356774245823|
|GATA1|0.361992081741905  |
|REST |0.14556435051563674|
+-----+-------------------+



In [16]:
# Aggregation 6 — Cell-type specificity
cell_type_summary = (
    hive_df.groupBy("TF", "cell_type")
      .agg(F.sum("num_peaks").alias("cell_type_peaks"))
)

cell_type_summary.show()

+-----+---------+---------------+
|   TF|cell_type|cell_type_peaks|
+-----+---------+---------------+
|GATA1|     K562|         177885|
| REST|  GM12878|          49876|
|GATA1|  GM12878|          81234|
| CTCF|  GM12878|         113221|
| REST|     K562|          54321|
| CTCF|     K562|         239277|
+-----+---------+---------------+



In [17]:
# Using Spark SQL instead of PySpark API
spark.sql("""
    SELECT
        TF,
        cell_type,
        AVG(num_peaks) AS avg_peaks
    FROM chipseq_metadata
    GROUP BY TF, cell_type
    ORDER BY TF, cell_type
""").show()

+-----+---------+---------+
|   TF|cell_type|avg_peaks|
+-----+---------+---------+
| CTCF|  GM12878| 113221.0|
| CTCF|     K562| 119638.5|
|GATA1|  GM12878|  81234.0|
|GATA1|     K562|  88942.5|
| REST|  GM12878|  49876.0|
| REST|     K562|  54321.0|
+-----+---------+---------+



In [18]:
spark.stop()